## 0. 환경설정 — 스드메 통합 챗봇

하나의 채팅방에서 스튜디오/드레스/메이크업 질문을 자유롭게 할 수 있습니다.

In [1]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python gradio python-dotenv -q

In [2]:
import os, json, re, time
from dotenv import load_dotenv

load_dotenv()

from neo4j import GraphDatabase, basic_auth
from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG
from openai import OpenAI
import gradio as gr

print("모듈 로드 완료")

c:\Users\SSAFY\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모듈 로드 완료


In [3]:
client = OpenAI()

## 1. Neo4j 연결 (읽기 전용 — 데이터 적재는 db_load.py)

In [4]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PW", "password123")

driver = GraphDatabase.driver(NEO4J_URI, auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    cnt = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    print(f"Neo4j 현재 노드 수: {cnt}")

Neo4j 현재 노드 수: 15959


In [5]:
# 카테고리별 데이터 존재 확인
with driver.session() as session:
    for cat in ["studio", "dress", "makeup"]:
        cnt = session.run(
            "MATCH (v:Vendor {category: $cat}) RETURN count(v) AS cnt", cat=cat
        ).single()["cnt"]
        status = f"{cnt}개 확인" if cnt > 0 else "⚠ 0개 — db_load.py 실행 필요"
        print(f"  {cat}: {status}")

  studio: 207개 확인
  dress: 268개 확인
  makeup: 472개 확인


## 2. MySQL (더미 fallback)

In [6]:
import mysql.connector

MYSQL_HOST = os.environ.get("MYSQL_HOST", "")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
COUPLE_ID = 2

conn = None
if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST, user=MYSQL_USER,
            password=MYSQL_PASSWORD, database=MYSQL_DB,
            port=int(os.environ.get("MYSQL_PORT", 3306))
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 - 더미 사용")

def get_user_preference(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_preferences WHERE couple_id = %s", (couple_id,))
            row = cur.fetchone(); cur.close()
            return row
        except: pass
    return {"couple_id": 2, "region": "서울", "sub_region": "강남구",
            "studio_style": "인물중심", "dress_style": "클래식", "makeup_style": "내추럴"}

def get_user_likes(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_venue_likes WHERE couple_id = %s", (couple_id,))
            rows = cur.fetchall(); cur.close()
            return rows
        except: pass
    return [{"name": "더미업체", "category": "스튜디오"}]

print("사용자 함수 정의 완료")

MySQL 연결 성공
사용자 함수 정의 완료


## 3. Neo4j 스키마 추출

In [7]:
def get_schema(uri, user, password):
    d = GraphDatabase.driver(uri, auth=basic_auth(user, password))
    with d.session() as session:
        nodes = session.run("MATCH (n) WITH DISTINCT labels(n) AS lbls, keys(n) AS ks, n UNWIND lbls AS l UNWIND ks AS k RETURN l, k, n[k] AS sv")
        rels = session.run("MATCH (a)-[r]->(b) RETURN DISTINCT labels(a) AS sl, type(r) AS rt, labels(b) AS el")
        schema = {"nodes": {}, "relations": []}
        for rec in nodes:
            l, k = rec["l"], rec["k"]
            if l not in schema["nodes"]: schema["nodes"][l] = {}
            v = rec["sv"]
            t = "STRING" if isinstance(v, str) else "INTEGER" if isinstance(v, int) else "FLOAT" if isinstance(v, float) else "UNKNOWN"
            schema["nodes"][l][k] = t
        for rec in rels:
            sl = rec["sl"][0] if rec["sl"] else "?"
            el = rec["el"][0] if rec["el"] else "?"
            schema["relations"].append(f"(:{sl})-[:{rec['rt']}]->(:{el})")
    d.close()
    return schema

def format_schema(schema):
    lines = ["Node properties:"]
    for label, props in schema["nodes"].items():
        ps = ", ".join(f"{k}: {v}" for k, v in props.items())
        lines.append(f"  {label} {{{ps}}}")
    lines.append("Relationships:")
    for r in schema["relations"]: lines.append(f"  {r}")
    return "\n".join(lines)

schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
neo4j_schema = format_schema(schema)
print(neo4j_schema)

Node properties:
  Review {name: STRING, date: STRING, score: INTEGER, contents: STRING}
  Vendor {likeCnt: INTEGER, iweddingNo: STRING, productPrice: INTEGER, category: STRING, profile: STRING, eventOptionPrice: INTEGER, orderCnt: INTEGER, detailCmt: STRING, enterpriseCode: STRING, salePrice: INTEGER, holiday: STRING, viewCnt: INTEGER, eventPrice: INTEGER, subCategory: STRING, reviewCnt: INTEGER, profileUrl: STRING, rating: FLOAT, productName: STRING, region: STRING, coverUrl: STRING, typeName: STRING, tel: STRING, partnerId: INTEGER, name: STRING, address: STRING, uuid: STRING}
  Region {name: STRING}
  Tag {category: STRING, name: STRING, typeName: STRING, type: INTEGER}
  StyleFilter {name: STRING, partnerType: INTEGER, id: INTEGER}
  Package {desc: STRING, title: STRING, value: STRING}
  Hall {minRentalPrice: INTEGER, rating: FLOAT, reviewCnt: INTEGER, availableContract: INTEGER, profileUrl: STRING, address: STRING, maxIndividualHallPrice: INTEGER, tel: STRING, region: STRING, nam

## 4. Few-shot Cypher 예시 (3개 카테고리 통합)

스튜디오/드레스/메이크업 예시가 모두 포함되어 있어, GraphRAG가 질문에서 카테고리를 자동 판별합니다.

In [8]:
examples = [
    # ============================================
    # 스튜디오 (category: 'studio')
    # ============================================
    """USER INPUT: '스튜디오 추천해줘'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '200만원 이하 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.salePrice <= 2000000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '야외씬 잘 찍는곳'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '야외' OR t.name CONTAINS '로드' OR t.name CONTAINS '가든'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '인물중심 스타일 추천'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '인물중심'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '강남 스튜디오 150만원 이하'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:IN_REGION]->(r:Region), (v)-[:HAS_TAG]->(t:Tag)
WHERE r.name CONTAINS '강남' AND v.salePrice <= 1500000 AND v.salePrice > 0
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '줄리의정원 가격이랑 패키지 알려줘'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.name CONTAINS '줄리의정원'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '줄리의정원이랑 식스플로어 비교해줘'
QUERY:
MATCH (v:Vendor {category:'studio'})
WHERE v.name CONTAINS '줄리의정원' OR v.name CONTAINS '식스플로어'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, collect(DISTINCT t.name) AS tags, avg(rv.score) AS avgScore, count(rv) AS revCnt
RETURN v.name AS name, v.salePrice AS price, v.rating AS rating, tags, round(avgScore, 1) AS avgScore, revCnt
ORDER BY v.name""",

    """USER INPUT: '줄리의정원과 비슷한 스타일'
QUERY:
MATCH (v1:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'studio'})
WHERE v1.name CONTAINS '줄리의정원' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '야외씬이랑 어울리는 스타일은?'
QUERY:
MATCH (t1:Tag {category:'studio'})-[c:CO_OCCURS]-(t2:Tag {category:'studio'})
WHERE t1.name CONTAINS '야외'
RETURN t2.name AS relatedTag, c.count AS freq
ORDER BY freq DESC LIMIT 10""",

    """USER INPUT: '리뷰 좋은 강남 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 드레스 (category: 'dress')
    # ============================================
    """USER INPUT: '드레스 추천해줘'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '200만원 이하 드레스'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.salePrice <= 2000000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '촬영+본식 드레스 4벌 이상'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '4벌' OR t.name CONTAINS '5벌'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '컬러 드레스 있는 곳'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '컬러'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '셀렉션H 가격이랑 구성 알려줘'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.name CONTAINS '셀렉션'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '셀렉션H와 비슷한 스타일 드레스샵'
QUERY:
MATCH (v1:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'dress'})
WHERE v1.name CONTAINS '셀렉션' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '리뷰 좋은 강남 드레스샵'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 메이크업 (category: 'makeup')
    # ============================================
    """USER INPUT: '메이크업 추천해줘'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '50만원 이하 메이크업'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.salePrice <= 500000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '내추럴 메이크업 추천'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '내추럴' OR t.name CONTAINS '깨끗'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '실장급 메이크업'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '실장'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '르보청담 가격이랑 구성 알려줘'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.name CONTAINS '르보'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '르보청담과 비슷한 스타일 메이크업샵'
QUERY:
MATCH (v1:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'makeup'})
WHERE v1.name CONTAINS '르보' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '리뷰 좋은 강남 메이크업샵'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 공통 패턴 (카테고리 무관)
    # ============================================
    """USER INPUT: '요즘 인기있는 곳'
QUERY:
MATCH (v:Vendor) WHERE v.orderCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.orderCnt AS orders, v.profileUrl AS url
ORDER BY v.orderCnt DESC LIMIT 10""",

    """USER INPUT: '가장 저렴한 곳 5개'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.salePrice ASC LIMIT 5""",

    """USER INPUT: '패키지 구성 알려줘'
QUERY:
MATCH (v:Vendor)-[:HAS_PACKAGE]->(p:Package)
WHERE v.name CONTAINS $vendorName
RETURN v.name AS name, v.category AS category, p.title AS packageTitle, p.value AS packageValue, p.desc AS packageDesc""",

    """USER INPUT: '비슷한 가격대 같은 지역'
QUERY:
MATCH (v1:Vendor)-[:IN_REGION]->(r:Region)<-[:IN_REGION]-(v2:Vendor)
WHERE v1.name CONTAINS $vendorName AND v1 <> v2 AND v1.category = v2.category
  AND abs(v2.salePrice - v1.salePrice) < 300000
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, r.name AS region
ORDER BY v2.rating DESC LIMIT 5""",
]

print(f"Few-shot 예시 {len(examples)}개 로드 완료 (스튜디오/드레스/메이크업 통합)")

Few-shot 예시 28개 로드 완료 (스튜디오/드레스/메이크업 통합)


## 5. LLM / Retriever / GraphRAG

In [9]:
llm = OpenAILLM(model_name="gpt-4o", model_params={"temperature": 0, "max_tokens": 2000})
retriever = Text2CypherRetriever(driver=driver, llm=llm, neo4j_schema=neo4j_schema, examples=examples)
rag = GraphRAG(retriever=retriever, llm=llm)
print("GraphRAG 초기화 완료")

GraphRAG 초기화 완료


## 6. Gradio 챗봇

In [ ]:
def build_query_with_context(message, chat_history):
    """최근 8개 메시지(4턴)를 포함하여 맥락 있는 쿼리 생성"""
    recent = chat_history[-8:] if chat_history else []
    if not recent:
        return message
    context = "\n".join(f"{m['role']}: {m['content'][:200]}" for m in recent)
    return f"[이전 대화]\n{context}\n\n[현재 질문]\n{message}"


def response_fn(message, chat_history):
    chat_history = chat_history or []
    msg = message.strip()

    # MySQL 연동이 필요한 intent만 규칙 기반 분기
    if any(x in msg for x in ["내 취향", "내 스타일", "내 선호"]):
        pref = get_user_preference(conn, COUPLE_ID)
        if pref:
            lines = [f"- {k}: {v}" for k, v in pref.items() if k != "couple_id" and v]
            answer = "현재 저장된 취향 정보입니다.\n" + "\n".join(lines)
        else:
            answer = "저장된 취향 정보가 없습니다."
    elif any(x in msg for x in ["좋아요한", "찜한", "찜 목록"]):
        likes = get_user_likes(conn, COUPLE_ID)
        if likes:
            lines = [f"- {l.get('name','알수없음')} ({l.get('category','')})" for l in likes]
            answer = f"좋아요한 업체가 {len(likes)}건 있습니다.\n" + "\n".join(lines)
        else:
            answer = "좋아요한 업체가 없습니다."
    else:
        # 맥락 포함 쿼리 생성 → GraphRAG에 위임
        query = build_query_with_context(message, chat_history)
        try:
            result = rag.search(query_text=query)
            answer = result.answer
        except Exception as e:
            print(f"[GraphRAG 오류] {e}")
            answer = "죄송합니다. 처리 중 오류가 발생했습니다. 다시 시도해주세요."

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": answer})
    return "", chat_history


with gr.Blocks(title="웨딩 스드메 추천 챗봇") as demo:
    gr.Markdown("# 웨딩 스드메 추천 챗봇\n스튜디오 / 드레스 / 메이크업 추천을 한 곳에서! (서울/경기)")
    chatbot = gr.Chatbot(height=600)
    with gr.Row():
        msg = gr.Textbox(placeholder="예: 200만원 이하 야외씬 스튜디오 추천해줘", show_label=False, scale=9)
        send_btn = gr.Button("전송", scale=1)
    gr.Markdown("### 이런 질문을 해보세요")
    with gr.Row():
        gr.Button("스튜디오 추천").click(lambda: ("스튜디오 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("드레스 추천").click(lambda: ("드레스 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("메이크업 추천").click(lambda: ("메이크업 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("야외씬 잘 찍는곳").click(lambda: ("야외씬 잘 찍는곳 추천해줘", None), outputs=[msg, chatbot])
    with gr.Row():
        gr.Button("촬영+본식 드레스").click(lambda: ("촬영+본식 드레스 4벌 이상 추천", None), outputs=[msg, chatbot])
        gr.Button("내추럴 메이크업").click(lambda: ("내추럴 메이크업 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("리뷰 좋은 곳").click(lambda: ("리뷰 좋은 곳 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("내 취향 보기").click(lambda: ("내 취향 알려줘", None), outputs=[msg, chatbot])
    msg.submit(response_fn, [msg, chatbot], [msg, chatbot])
    send_btn.click(response_fn, [msg, chatbot], [msg, chatbot])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/03/16 15:22:09 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


[GraphRAG 오류] Text2CypherRetriever.get_search_results() got an unexpected keyword argument 'top_k'
